# <center> Кластеризация изображений транспортных средств

## Постановка задачи

<center> <img src=https://i.ibb.co/t8DvkyB/smart-city-image-1.jpg align="right" width="300"/> </center>
<center> <img src=https://i.ibb.co/qYkWNVh/smart-city-image-3.jpg align="right" width="300"/> </center>


Один из ключевых проектов IntelliVision — Smart City/Transportation, система, обеспечивающая безопасность дорожного движения и более эффективную работу парковок. С помощью Smart City/Transportation можно контролировать сигналы светофоров и соблюдение ограничений скорости, определять виды транспортных средств, распознавать номерные знаки, считать автомобили и людей.

В основе всех перечисленных возможностей проекта лежит CV (Computer Vision, компьютерное зрение). Чтобы их реализовать, компания использует модели, для обучения которых применяются огромные размеченные датасеты с изображениями транспортных средств. Однако система работает в режиме реального времени и с каждым днём данных становится всё больше. Алгоритм нуждается в постоянной модернизации и должен учитывать множество факторов.

Для модификации и повышения эффективности системы Smart City/Transportation команде необходимо автоматизировать определение дополнительных параметров авто на изображении:

* тип автомобиля (кузова),
* ракурс снимка (вид сзади/спереди),
* цвет автомобиля,
* другие характеристики.

Также необходимо автоматизировать поиск выбросов в данных (засветы и блики на изображениях, изображения, на которых отсутствуют автомобили и т. д.).

К сожалению, у компании нет комплексной модели, которая могла бы одновременно находить на изображении автомобиль и определять все нужные параметры. Её нужно построить, однако многокомпонентная разметка новых данных по всем этим параметрам — очень трудозатратное занятие, которое стоит больших денег.

При решении задачи разметки данных у команды возникла гипотеза, которая нуждается в исследовании.


**Гипотеза:** разметку исходных данных можно эффективно провести с помощью методов кластеризации. 


**В чём идея?**

*Давайте будем использовать небольшой набор моделей свёрточных нейронных сетей, обученных на различных датасетах и решающих различные задачи от классификации изображений по цвету до классификации типов транспортных средств, пропустим нашу базу изображений через каждую модель, но возьмём не выходной результат модели, а только промежуточное представление признаков (дескриптор), полученное на свёрточных слоях сети.*

*Выполним такую операцию для всех изображений из набора данных, на основе полученных дескрипторов кластеризуем изображения, проинтерпретируем полученные кластеры и попробуем найти в них необходимую информацию.*

Теперь, когда мы обсудили гипотезу, перейдём к постановке задачи.

<center> <img src=https://i.ibb.co/hLcBpZF/2023-03-27-12-11-17.png align="right" width="500"/> </center>

У нас будет набор из 416 314 изображений транспортных средств различных типов, цветов и снятых с разных ракурсов.

Команда IntelliVision уже обработала свой набор данных с помощью нескольких моделей глубокого обучения (свёрточных нейронных сетей) и получила четыре варианта вектора признаков (дескрипторов) для каждого изображения.

**Наша задача** — используя готовые дескрипторы, разбить изображения на кластеры и проинтерпретировать каждый из них. Для всех вариантов дескрипторов нужно применить несколько алгоритмов кластеризации и сравнить полученные результаты. Сравнивать можно на основе метрик, визуализаций плотностей кластеров и по тому, насколько хорошо интерпретируются кластеры.

Дополнительная подзадача — найти выбросы среди изображений. Это могут быть изображения плохого качества, изображения с бликами или изображения, на которых нет транспортных средств и т. д.

Бизнес-задача: исследовать возможность применения алгоритмов кластеризации для разметки новых данных и поиска выбросов.

Техническая задача для нас как для специалиста в Data Science: построить модель кластеризации изображений на основе дескрипторов, выделяемых с помощью различных архитектур нейронных сетей, проинтерпретировать полученные результаты и выбрать модель или комбинацию моделей, которая выделяет наиболее пригодные для интерпретации признаки.

**Наши основные цели:**
1. Для каждого типа дескрипторов необходимо:
    * выполнить предобработку дескрипторов;
    * произвести кластеризацию изображений на основе их дескрипторов, подобрав алгоритм и параметры кластеризации;
    * сделать визуализацию полученных кластеров в 2D- или 3D-пространстве;
    * проинтерпретировать полученные кластеры — в паре предложений сформулировать, какие изображения попали в каждый из кластеров.
2. Сравнить между собой полученные кластеризации для каждого типа дескрипторов (по метрикам, визуализации и результатам интерпретации).
3. Выполнить автоматизированный поиск выбросов среди изображений на основе дескрипторов.
4. Дополнительная задача: попробовать воспользоваться смесью дескрипторов, полученных различными моделями, и проинтерпретировать полученные результаты.

**Примечание.** При выборе алгоритма кластеризации будем ориентироваться на внутренние метрики, а именно на индекс Калински — Харабаса (`calinski_harabasz_score`) и индекс Дэвиса — Болдина (`davies_bouldin_score`), а также на интерпретируемость кластеров и визуализацию.

## Данные и их описание

Исходная папка с данными имеет следующую структуру:

```
IntelliVision_case
├─descriptors
    └─efficientnet-b7.pickle
    └─osnet.pickle
    └─vdc_color.pickle
    └─vdc_type.pickle
├─row_data
    └─veriwild.zip
├─images_paths.csv 
```

Давайте разберёмся в ней:

* В папке `descriptors` содержатся дескрипторы, полученные для каждого из изображений с помощью соответствующих нейронных сетей, в формате numpy-массивов, сохранённых в файлах pickle:
    * `efficientnet-b7.pickle` — дескрипторы, выделенные моделью классификации с архитектурой EfficientNet версии 7. Эта модель является свёрточной нейронной сетью, предобученной на на датасете ImageNet, в котором содержатся изображения более 1000 различных классов. Эта модель при обучении не видела датасета veriwiId. 

    * `osnet.pickle` — дескрипторы, выделенные моделью OSNet, обученной для детектирования людей, животных и машин. Модель не обучалась на исходном датасете veriwiId.

    * `vdc_color.pickle` — дескрипторы, выделенные моделью регрессии для определения цвета транспортных средств в формате RGB. Частично обучена на исходном датасете veriwild.
    
    * `vdc_type.pickle` — дескрипторы, выделенные моделью классификации транспортных средств по типу на десяти классах. Частично обучена на исходном датасете veriwild.

* В папке `row_data` содержится zip-архив с исходными изображениями автомобилей. Распакуйте его содержимое в папку row_data. Архив содержит десять папок с изображениями, пронумерованных от 1 до 10. Каждая папка содержит подпапки, обозначенные пятизначными цифрами, например 36191. 

В каждой из таких подпапок содержатся фотографии одного конкретного автомобиля с разных ракурсов, снятые с помощью дорожных видеокамер.

* В файле `images_paths.csv` представлен список из полных путей до изображений. Он пригодится вам при анализе изображений, попавших в определённый кластер.


## 1. Знакомство со структурой данных

Прочитаем numpy-массивы из предоставленных pickle-файлов.

Примечание Для удобства дальнейшей работы составим четыре DataFrame с путями до изображений и соответствующими им дескрипторами.

Посмотрим на размерности каждой из четырёх заданных матриц и сравните использованные модели глубокого обучения по размерностям выходных дескрипторов изображений.

In [1]:
# Базовый путь к датасету
BASE_PATH = '/kaggle/input/datasets/markhomeless/intellivision-case/IntelliVision_case'

# Пути к дескрипторам
efficientnet_path = f'{BASE_PATH}/descriptors/efficientnet-b7.pickle'
osnet_path = f'{BASE_PATH}/descriptors/osnet.pickle'
vdc_color_path = f'{BASE_PATH}/descriptors/vdc_color.pickle'
vdc_type_path = f'{BASE_PATH}/descriptors/vdc_type.pickle'
# Путь к CSV с путями изображений
images_paths = f'{BASE_PATH}/images_paths.csv'

In [2]:
# Загружаем пути к изображениям
import pandas as pd
import pickle

paths_df = pd.read_csv(images_paths)

import os
# Функция для загрузки дескрипторов
def load_descriptors(file_path, name):
    print(f"\nЗагрузка {name}...")
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            descriptors = pickle.load(f)
        print(f"  Файл: {file_path}")
        print(f"  Форма массива: {descriptors.shape}")
        print(f"  Тип данных: {descriptors.dtype}")
        return descriptors
    else:
        print(f"  Файл НЕ НАЙДЕН: {file_path}")
        return None

# Загружаем все дескрипторы
X_ef = load_descriptors(efficientnet_path, "EfficientNet")
X_osnet = load_descriptors(osnet_path, "OSNet")
X_color = load_descriptors(vdc_color_path, "VDC Color")
X_type = load_descriptors(vdc_type_path, "VDC Type")


Загрузка EfficientNet...
  Файл: /kaggle/input/datasets/markhomeless/intellivision-case/IntelliVision_case/descriptors/efficientnet-b7.pickle
  Форма массива: (416314, 2560)
  Тип данных: float32

Загрузка OSNet...
  Файл: /kaggle/input/datasets/markhomeless/intellivision-case/IntelliVision_case/descriptors/osnet.pickle
  Форма массива: (416314, 512)
  Тип данных: float32

Загрузка VDC Color...
  Файл: /kaggle/input/datasets/markhomeless/intellivision-case/IntelliVision_case/descriptors/vdc_color.pickle
  Форма массива: (416314, 128)
  Тип данных: float32

Загрузка VDC Type...
  Файл: /kaggle/input/datasets/markhomeless/intellivision-case/IntelliVision_case/descriptors/vdc_type.pickle
  Форма массива: (416314, 512)
  Тип данных: float32


## 2. Преобразование, очистка и анализ данных

Признаки, найденные с помощью некоторых моделей, исчисляются тысячами, что довольно много, учитывая общее количество наблюдений.

Как вы понимаете, производить кластеризацию на таком большом количестве признаков, которые были сформированы исходными моделями глубокого обучения, довольно сложно и затратно по времени. К тому же, многие признаки, найденные моделями на изображениях, могут быть сильно скоррелированы между собой.

Будем понижать размерность исходных дескрипторов с помощью соответствующих методов. Можно уменьшить размерность входных данных до 100 или 200 признаков — этого будет достаточно, чтобы произвести кластеризацию. Попробуем подобрать необходимое количество компонент в новом пространстве признаков.

Для работы в общем масштабе признаков, воспользуемся стандартизацией и нормализацией.

In [3]:
# Предобработка данных: Стандартизация и уменьшение размерности
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Создадим словарь с дескрипторами для удобной обработки
descriptors_dict = {
    'EfficientNet': X_ef,
    'OSNet': X_osnet,
    'VDC_Color': X_color,
    'VDC_Type': X_type
}

# Словарь для хранения обработанных данных
processed_descriptors = {}

# Обрабатываем каждый тип дескрипторов
for name, X in descriptors_dict.items():
    print(f"\nОбработка {name}...")
    print(f"  Исходная размерность: {X.shape}")
    
    # 1. Стандартизация
    print("  Шаг 1: Стандартизация (StandardScaler)")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # 2. Уменьшение размерности до 100 компонент (PCA)
    print("  Шаг 2: Уменьшение размерности PCA до 100 компонент")
    pca = PCA(n_components=100, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    
    # Сохраняем обработанные данные
    processed_descriptors[name] = X_pca
    print(f"  Результат: {X_pca.shape}")
    print(f"  Объясненная дисперсия: {pca.explained_variance_ratio_.sum():.2%}")

print("\n" + "="*60)
print("ПРЕДОБРАБОТКА ЗАВЕРШЕНА")
print("="*60)


Обработка EfficientNet...
  Исходная размерность: (416314, 2560)
  Шаг 1: Стандартизация (StandardScaler)
  Шаг 2: Уменьшение размерности PCA до 100 компонент
  Результат: (416314, 100)
  Объясненная дисперсия: 46.15%

Обработка OSNet...
  Исходная размерность: (416314, 512)
  Шаг 1: Стандартизация (StandardScaler)
  Шаг 2: Уменьшение размерности PCA до 100 компонент
  Результат: (416314, 100)
  Объясненная дисперсия: 84.06%

Обработка VDC_Color...
  Исходная размерность: (416314, 128)
  Шаг 1: Стандартизация (StandardScaler)
  Шаг 2: Уменьшение размерности PCA до 100 компонент
  Результат: (416314, 100)
  Объясненная дисперсия: 97.07%

Обработка VDC_Type...
  Исходная размерность: (416314, 512)
  Шаг 1: Стандартизация (StandardScaler)
  Шаг 2: Уменьшение размерности PCA до 100 компонент
  Результат: (416314, 100)
  Объясненная дисперсия: 97.66%

ПРЕДОБРАБОТКА ЗАВЕРШЕНА


### Анализ результатов PCA

Мы выполнили PCA для всех четырех типов дескрипторов, уменьшив размерность до 100 компонент. Полученные результаты показывают разную степень сжатия информации:

| Модель | Исходная размерность | Объясненная дисперсия (100 компонент) |
|--------|---------------------|--------------------------------------|
| EfficientNet | 2560 | 46.15% |
| OSNet | 512 | 84.06% |
| VDC_Color | 128 | 97.07% |
| VDC_Type | 512 | 97.66% |

**Что это значит:**
- Для EfficientNet 100 компонент недостаточно (только 46% информации)
- Для VDC_Color и VDC_Type можно взять меньше компонент (информация очень сжата)
- Для OSNet 100 компонент дают хороший результат (84%)

**Следующий шаг:** подберем оптимальное количество компонент для каждого типа дескрипторов (чтобы сохранить 95% дисперсии) и выполним финальную предобработку.

In [4]:
# Подбор оптимального количества компонент PCA и финальная предобработка
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Словарь для хранения оптимальных параметров
pca_params = {}
processed_descriptors = {}

for name, X in descriptors_dict.items():
    print(f"\nАнализ для {name} (исходная размерность: {X.shape[1]})...")
    
    # Стандартизация
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # PCA для анализа
    pca = PCA(random_state=42)
    pca.fit(X_scaled)
    
    # Считаем кумулятивную объясненную дисперсию
    cumsum = np.cumsum(pca.explained_variance_ratio_)
    
    # Находим количество компонент для 95% дисперсии
    n_components_95 = np.argmax(cumsum >= 0.95) + 1
    variance_95 = cumsum[n_components_95 - 1] * 100
    
    print(f"  Компонент для 95% дисперсии: {n_components_95}")
    print(f"  Фактическая дисперсия: {variance_95:.2f}%")
    
    # Сохраняем параметры
    pca_params[name] = n_components_95
    
    # Выполняем финальное PCA с оптимальным количеством компонент
    print(f"  Выполняем PCA до {n_components_95} компонент...")
    pca_final = PCA(n_components=n_components_95, random_state=42)
    X_pca = pca_final.fit_transform(X_scaled)
    
    processed_descriptors[name] = X_pca
    print(f"  Результат: {X_pca.shape}")
    print(f"  Объясненная дисперсия: {pca_final.explained_variance_ratio_.sum():.2%}")

print("\n" + "="*60)
print("ПРЕДОБРАБОТКА ЗАВЕРШЕНА")
print("="*60)
print("\nИтоговые размерности:")
for name, X in processed_descriptors.items():
    print(f"  {name}: {X.shape}")


Анализ для EfficientNet (исходная размерность: 2560)...
  Компонент для 95% дисперсии: 1807
  Фактическая дисперсия: 95.00%
  Выполняем PCA до 1807 компонент...
  Результат: (416314, 1807)
  Объясненная дисперсия: 94.71%

Анализ для OSNet (исходная размерность: 512)...
  Компонент для 95% дисперсии: 235
  Фактическая дисперсия: 95.00%
  Выполняем PCA до 235 компонент...
  Результат: (416314, 235)
  Объясненная дисперсия: 95.00%

Анализ для VDC_Color (исходная размерность: 128)...
  Компонент для 95% дисперсии: 90
  Фактическая дисперсия: 95.19%
  Выполняем PCA до 90 компонент...
  Результат: (416314, 90)
  Объясненная дисперсия: 95.19%

Анализ для VDC_Type (исходная размерность: 512)...
  Компонент для 95% дисперсии: 46
  Фактическая дисперсия: 95.03%
  Выполняем PCA до 46 компонент...
  Результат: (416314, 46)
  Объясненная дисперсия: 95.03%

ПРЕДОБРАБОТКА ЗАВЕРШЕНА

Итоговые размерности:
  EfficientNet: (416314, 1807)
  OSNet: (416314, 235)
  VDC_Color: (416314, 90)
  VDC_Type: (416

### Анализ результатов подбора оптимальной размерности

Мы выполнили PCA для каждого типа дескрипторов, подобрав количество компонент, сохраняющее 95% исходной информации.

**Итоговые размерности после предобработки:**

| Модель | Исходная размерность | Оптимальная размерность (95%) | Сжатие |
|--------|---------------------|-------------------------------|--------|
| EfficientNet | 2560 | 1807 | в 1.4 раза |
| OSNet | 512 | 235 | в 2.2 раза |
| VDC_Color | 128 | 90 | в 1.4 раза |
| VDC_Type | 512 | 46 | в 11.1 раза |

**Полученные результаты:**

1. **EfficientNet**: 2560 → 1807 признаков (95% информации)
2. **OSNet**: 512 → 235 признаков (95% информации)  
3. **VDC_Color**: 128 → 90 признаков (95% информации)
4. **VDC_Type**: 512 → 46 признаков (95% информации)

Дополнительные методы уменьшения размерности не требуются, так как задача решена — данные подготовлены для кластеризации с сохранением основного объема информации.

In [5]:
# Сохранение промежуточных результатов 2 этапа
import os
import shutil
import json

# Сохраняем в working
work_path = '/kaggle/working/intellivision/02'
os.makedirs(work_path, exist_ok=True)

with open(f'{work_path}/processed_descriptors.pickle', 'wb') as f:
    pickle.dump(processed_descriptors, f)
    
paths_df.to_csv(f'{work_path}/images_paths.csv', index=False)

for name, X in processed_descriptors.items():
    np.save(f'{work_path}/{name}_pca.npy', X)

# Проверяем наличие метаданных в корне
if not os.path.exists('/kaggle/working/intellivision/dataset-metadata.json'):
    !kaggle datasets init -p /kaggle/working/intellivision
    with open('/kaggle/working/intellivision/dataset-metadata.json', 'r') as f:
        metadata = json.load(f)
    metadata['id'] = 'tregubovsergei/intellivision'
    metadata['title'] = 'IntelliVision'
    with open('/kaggle/working/intellivision/dataset-metadata.json', 'w') as f:
        json.dump(metadata, f, indent=4)

# Загружаем на Kaggle
!kaggle datasets version -p /kaggle/working/intellivision -m "update 02" --dir-mode zip

# Чистим
shutil.rmtree('/kaggle/working/intellivision')
print("\n" + "="*60)
print("СОХРАНЕНО")
print("="*60)

Data package template written to: /kaggle/working/intellivision/dataset-metadata.json
Starting upload for file 02.zip
100%|███████████████████████████████████████| 6.31G/6.31G [00:55<00:00, 122MB/s]
Upload successful: 02.zip (6GB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/tregubovsergei/intellivision

СОХРАНЕНО


## 3. Моделирование и оценка качества модели

In [6]:
# Загрузка данных из сэйва
import pickle
import numpy as np
import pandas as pd
import os

if 'processed_descriptors' not in dir(): # список всех имён переменных в текущей сессии
    print('Загрузка данных из датасета...')
    
    data_path = '/kaggle/input/intellivision/02'
    
    with open(f'{data_path}/processed_descriptors.pickle', 'rb') as f:
        processed_descriptors = pickle.load(f)
    
    paths_df = pd.read_csv(f'{data_path}/images_paths.csv')
    
    pca_results = {}
    for name in processed_descriptors.keys():
        pca_results[name] = np.load(f'{data_path}/{name}_pca.npy')
    
    print('Данные загружены с датасета')
else:
    print('Данные в памяти, продолжаем')

Данные в памяти, продолжаем


### 3.1. Кластеризация изображений

После предобработки данных приступаем к кластеризации. Для каждого типа дескрипторов применим несколько алгоритмов кластеризации и подберем оптимальное количество кластеров.

**Используемые алгоритмы:**
- **K-Means** (через MiniBatchKMeans для больших данных)
- **EM-алгоритм** (Gaussian Mixture Model)
- **Агломеративная иерархическая кластеризация** (с ограничением по глубине)
- **DBSCAN** (для поиска кластеров произвольной формы)

**Метрики качества:**
- **Calinski-Harabasz Index** (чем выше, тем лучше)
- **Davies-Bouldin Index** (чем ниже, тем лучше)

Для ускорения работы будем использовать подвыборку для некоторых алгоритмов (EM, агломеративная кластеризация), так как они требуют много памяти на полных данных.

In [7]:
# Кластеризация изображений

from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import DBSCAN
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

# Словари для хранения результатов кластеризации
clustering_results = {}
cluster_labels = {}

# Количество кластеров для K-Means и EM (подбирается экспериментально)
# Для разных типов дескрипторов можно использовать разное количество кластеров
n_clusters = {
    'EfficientNet': 20,
    'OSNet': 15,
    'VDC_Color': 12,
    'VDC_Type': 10
}

# Применяем кластеризацию к каждому типу дескрипторов
for name, X in processed_descriptors.items():
    print(f"\n{'-'*60}")
    print(f"КЛАСТЕРИЗАЦИЯ ДЛЯ {name}")
    print(f"Размерность данных: {X.shape}")
    print(f"{'-'*60}")
    
    # Словарь для хранения результатов по данному типу дескрипторов
    results = {}
    labels_dict = {}
    
    # 1. MiniBatchKMeans (аналог K-Means для больших данных)
    print("\n1. MiniBatchKMeans...")
    k = n_clusters[name]
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=10000, random_state=42, n_init=3)
    labels_kmeans = kmeans.fit_predict(X)
    labels_dict['kmeans'] = labels_kmeans
    
    # Метрики
    ch_score = calinski_harabasz_score(X, labels_kmeans)
    db_score = davies_bouldin_score(X, labels_kmeans)
    results['kmeans'] = {
        'model': kmeans,
        'calinski_harabasz': ch_score,
        'davies_bouldin': db_score,
        'n_clusters': len(np.unique(labels_kmeans))
    }
    print(f"  Количество кластеров: {len(np.unique(labels_kmeans))}")
    print(f"  Индекс Калински-Харабаса: {ch_score:.2f}")
    print(f"  Индекс Дэвиса-Болдина: {db_score:.2f}")
    
    # 2. Gaussian Mixture Model (EM-алгоритм)
    print("\n2. Gaussian Mixture Model (EM-алгоритм)...")
    # Используем подвыборку для ускорения (10000 объектов)
    np.random.seed(42)
    idx_sample = np.random.choice(X.shape[0], size=min(20000, X.shape[0]), replace=False)
    X_sample = X[idx_sample]
    
    gmm = GaussianMixture(n_components=k, random_state=42, max_iter=100)
    labels_gmm_sample = gmm.fit_predict(X_sample)
    
    # Для полного настава предсказываем кластеры
    from scipy.spatial.distance import cdist
    
    # Предсказание для всех данных на основе близости к центрам компонент
    distances = cdist(X, gmm.means_)
    labels_gmm = np.argmin(distances, axis=1)
    labels_dict['gmm'] = labels_gmm
    
    # Метрики на полных данных
    ch_score = calinski_harabasz_score(X, labels_gmm)
    db_score = davies_bouldin_score(X, labels_gmm)
    results['gmm'] = {
        'model': gmm,
        'calinski_harabasz': ch_score,
        'davies_bouldin': db_score,
        'n_clusters': len(np.unique(labels_gmm))
    }
    print(f"  Количество кластеров: {len(np.unique(labels_gmm))}")
    print(f"  Индекс Калински-Харабаса: {ch_score:.2f}")
    print(f"  Индекс Дэвиса-Болдина: {db_score:.2f}")
    
    # 3. Агломеративная иерархическая кластеризация
    print("\n3. Agglomerative Clustering...")
    # Для иерархической кластеризации также используем подвыборку
    agg = AgglomerativeClustering(n_clusters=k)
    labels_agg_sample = agg.fit_predict(X_sample)
    
    # Для полного набора используем K-Means для назначения кластеров на основе центроидов
    # Находим центроиды кластеров на подвыборке
    centroids = np.zeros((k, X.shape[1]))
    for i in range(k):
        mask = labels_agg_sample == i
        if np.sum(mask) > 0:
            centroids[i] = np.mean(X_sample[mask], axis=0)
        else:
            centroids[i] = X_sample[np.random.randint(0, len(X_sample))]
    
    # Назначаем кластеры для всех объектов по ближайшему центроиду
    distances = cdist(X, centroids)
    labels_agg = np.argmin(distances, axis=1)
    labels_dict['agg'] = labels_agg
    
    # Метрики
    ch_score = calinski_harabasz_score(X, labels_agg)
    db_score = davies_bouldin_score(X, labels_agg)
    results['agg'] = {
        'model': agg,
        'calinski_harabasz': ch_score,
        'davies_bouldin': db_score,
        'n_clusters': len(np.unique(labels_agg))
    }
    print(f"  Количество кластеров: {len(np.unique(labels_agg))}")
    print(f"  Индекс Калински-Харабаса: {ch_score:.2f}")
    print(f"  Индекс Дэвиса-Болдина: {db_score:.2f}")
    
    # 4. DBSCAN (для поиска выбросов и кластеров произвольной формы)
    print("\n4. DBSCAN...")
    # Подбираем параметры эмпирически (eps - радиус окрестности, min_samples - мин. точек в кластере)
    # Используем подвыборку для подбора параметров
    dbscan = DBSCAN(eps=0.5, min_samples=10, n_jobs=-1)
    labels_dbscan_sample = dbscan.fit_predict(X_sample)
    
    # Количество кластеров (исключая шум)
    n_clusters_db = len(set(labels_dbscan_sample)) - (1 if -1 in labels_dbscan_sample else 0)
    n_noise = list(labels_dbscan_sample).count(-1)
    
    # Для полных данных используем тот же подход с центроидами
    unique_labels = np.unique(labels_dbscan_sample)
    valid_labels = unique_labels[unique_labels != -1]
    
    if len(valid_labels) > 0:
        # Находим центроиды для кластеров (исключая шум)
        centroids_db = np.zeros((len(valid_labels), X.shape[1]))
        for i, label in enumerate(valid_labels):
            mask = labels_dbscan_sample == label
            centroids_db[i] = np.mean(X_sample[mask], axis=0)
        
        # Назначаем кластеры для всех объектов
        distances = cdist(X, centroids_db)
        labels_dbscan = np.argmin(distances, axis=1)
        
        # Точки, которые были шумом в подвыборке, оставляем как отдельный кластер?
        # Добавим их как отдельную метку (-1)
        # Для простоты будем считать, что все точки относятся к ближайшему кластеру
    else:
        # Если кластеров не найдено, все точки - шум
        labels_dbscan = np.full(X.shape[0], -1)
    
    labels_dict['dbscan'] = labels_dbscan
    
    # Метрики (только для нешумовых точек, если их достаточно)
    mask_non_noise = labels_dbscan != -1
    if np.sum(mask_non_noise) > 100:  # Если достаточно нешумовых точек
        ch_score = calinski_harabasz_score(X[mask_non_noise], labels_dbscan[mask_non_noise])
        db_score = davies_bouldin_score(X[mask_non_noise], labels_dbscan[mask_non_noise])
    else:
        ch_score = 0
        db_score = 999
    
    results['dbscan'] = {
        'model': dbscan,
        'calinski_harabasz': ch_score,
        'davies_bouldin': db_score,
        'n_clusters': n_clusters_db,
        'n_noise': n_noise
    }
    print(f"  Количество кластеров (без шума): {n_clusters_db}")
    print(f"  Количество шумовых точек: {n_noise} ({n_noise/len(X_sample)*100:.1f}%)")
    if ch_score > 0:
        print(f"  Индекс Калински-Харабаса: {ch_score:.2f}")
        print(f"  Индекс Дэвиса-Болдина: {db_score:.2f}")
    
    # Сохраняем результаты
    clustering_results[name] = results
    cluster_labels[name] = labels_dict
    
    print(f"\nКластеризация для {name} завершена")

print("\n" + "="*60)
print("КЛАСТЕРИЗАЦИЯ ЗАВЕРШЕНА ДЛЯ ВСЕХ ТИПОВ ДЕСКРИПТОРОВ")
print("="*60)


------------------------------------------------------------
КЛАСТЕРИЗАЦИЯ ДЛЯ EfficientNet
Размерность данных: (416314, 1807)
------------------------------------------------------------

1. MiniBatchKMeans...
  Количество кластеров: 20
  Индекс Калински-Харабаса: 2741.69
  Индекс Дэвиса-Болдина: 4.69

2. Gaussian Mixture Model (EM-алгоритм)...
  Количество кластеров: 20
  Индекс Калински-Харабаса: 3049.40
  Индекс Дэвиса-Болдина: 6.58

3. Agglomerative Clustering...
  Количество кластеров: 20
  Индекс Калински-Харабаса: 3013.58
  Индекс Дэвиса-Болдина: 6.83

4. DBSCAN...
  Количество кластеров (без шума): 0
  Количество шумовых точек: 20000 (100.0%)

Кластеризация для EfficientNet завершена

------------------------------------------------------------
КЛАСТЕРИЗАЦИЯ ДЛЯ OSNet
Размерность данных: (416314, 235)
------------------------------------------------------------

1. MiniBatchKMeans...
  Количество кластеров: 15
  Индекс Калински-Харабаса: 11831.01
  Индекс Дэвиса-Болдина: 3.06

## Выводы по кластеризации изображений транспортных средств

### 1. Сравнение качества кластеризации по типам дескрипторов

| Модель | Алгоритм | Индекс Калински-Харабаса (↑) | Индекс Дэвиса-Болдина (↓) |
|--------|----------|------------------------------|---------------------------|
| **EfficientNet** | K-Means | 2 741.69 | 4.69 |
| | GMM | 3 049.40 | 6.58 |
| | Agglomerative | 3 013.58 | 6.83 |
| **OSNet** | K-Means | 11 831.01 | 3.06 |
| | GMM | 12 249.89 | 3.00 |
| | Agglomerative | 12 281.62 | 2.73 |
| **VDC_Color** | K-Means | 29 558.04 | 2.66 |
| | GMM | 28 832.36 | 2.68 |
| | Agglomerative | 29 647.74 | 2.53 |
| **VDC_Type** | K-Means | 69 627.42 | 1.73 |
| | GMM | 63 554.92 | 1.88 |
| | Agglomerative | 70 379.78 | 1.55 |

### 2. Анализ полученных результатов

**Наилучшие результаты демонстрирует VDC_Type:**
- Самый высокий индекс Калински-Харабаса (>70 000) - кластеры максимально компактны и хорошо разделены
- Самый низкий индекс Дэвиса-Болдина (1.55) - минимальное внутрикластерное расстояние и максимальное межкластерное
- Это ожидаемо, так как модель обучалась на классификацию типов транспортных средств (10 классов)

**VDC_Color показывает хорошие результаты:**
- Второй по качеству кластеризации
- Модель обучалась на определение цвета в формате RGB, что обеспечивает естественное разделение изображений по цветовым характеристикам

**OSNet демонстрирует средние результаты:**
- Индекс Калински-Харабаса ~12 000
- Модель обучалась на детектирование людей, животных и машин, поэтому выделяет более общие признаки

**EfficientNet показывает наихудшие результаты:**
- Самые низкие метрики качества
- Модель обучалась на ImageNet (1000 общих классов) и не видела датасет veriwild, поэтому дескрипторы не оптимизированы для транспортных средств

### 3. Сравнение алгоритмов кластеризации

**Agglomerative clustering:**
- Показал лучшие метрики для VDC_Type и VDC_Color
- Индекс Дэвиса-Болдина 1.55 (лучший результат)
- Требователен к вычислительным ресурсам

**K-Means (MiniBatchKMeans):**
- Близкие к оптимальным метрики
- Значительно быстрее агломеративной кластеризации
- Хороший выбор для практического применения

**GMM (EM-алгоритм):**
- Результаты близки к K-Means
- Немного уступает по компактности кластеров

**DBSCAN:**
- Не нашел кластеров при параметрах eps=0.5, min_samples=10
- Все точки определены как шум (100%)
- Это говорит о высокой плотности данных и отсутствии естественных "разрывов" между кластерами
- Требует дополнительного подбора параметров

### 4. Общие выводы

1. **Гипотеза подтвердилась** - кластеризация на основе дескрипторов позволяет эффективно группировать изображения транспортных средств

2. **VDC_Type наиболее пригоден для интерпретации** - модель, обученная на типах авто, создает наиболее естественные кластеры

3. **Для практического применения рекомендуется:**
   - Использовать дескрипторы VDC_Type (тип авто) и VDC_Color (цвет)
   - Применять алгоритм K-Means как оптимальный по соотношению качество/скорость
   - Для более точного анализа использовать агломеративную кластеризацию

4. **DBSCAN неэффективен для данных дескрипторов** при стандартных параметрах - данные слишком плотные

5. **Размерность дескрипторов влияет на качество** - чем специфичнее модель (VDC_Type, VDC_Color), тем лучше кластеризация даже при меньшей размерности признаков

In [8]:
# Сохранение результатов кластеризации
import os
import shutil
import json

# Сохраняем в working
work_path = '/kaggle/working/intellivision/03'
os.makedirs(work_path, exist_ok=True)

# Сохраняем результаты кластеризации
with open(f'{work_path}/clustering_results.pickle', 'wb') as f:
    pickle.dump(clustering_results, f)
    
with open(f'{work_path}/cluster_labels.pickle', 'wb') as f:
    pickle.dump(cluster_labels, f)

# Метаданные (если ещё нет)
if not os.path.exists('/kaggle/working/intellivision/dataset-metadata.json'):
    !kaggle datasets init -p /kaggle/working/intellivision
    with open('/kaggle/working/intellivision/dataset-metadata.json', 'r') as f:
        metadata = json.load(f)
    metadata['id'] = 'tregubovsergei/intellivision'
    metadata['title'] = 'IntelliVision'
    with open('/kaggle/working/intellivision/dataset-metadata.json', 'w') as f:
        json.dump(metadata, f, indent=4)

# Загружаем на Kaggle (02 остаётся нетронутой)
!kaggle datasets version -p /kaggle/working/intellivision -m "add clustering results to /03" --dir-mode zip

# Чистим
shutil.rmtree('/kaggle/working/intellivision')
print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ КЛАСТЕРИЗАЦИИ СОХРАНЕНЫ")
print("="*60)

Data package template written to: /kaggle/working/intellivision/dataset-metadata.json
Starting upload for file 03.zip
100%|███████████████████████████████████████| 1.19G/1.19G [00:07<00:00, 181MB/s]
Upload successful: 03.zip (1GB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/tregubovsergei/intellivision

РЕЗУЛЬТАТЫ КЛАСТЕРИЗАЦИИ СОХРАНЕНЫ


In [9]:
# Загрузка данных после кластеризации
import pickle
import numpy as np
import pandas as pd
import os

data_path = '/kaggle/input/intellivision'
# Загрузка дескрипторов (если нужны)

if 'processed_descriptors' not in dir(): 
    if os.path.exists(f'{data_path}/02/processed_descriptors.pickle'):
        with open(f'{data_path}/02/processed_descriptors.pickle', 'rb') as f:
            processed_descriptors = pickle.load(f)
        
        paths_df = pd.read_csv(f'{data_path}/02/images_paths.csv')
        
        pca_results = {}
        for name in processed_descriptors.keys():
            pca_results[name] = np.load(f'{data_path}/02/{name}_pca.npy')
        
        print('Дискрипторы загружены из датасета')
    else:
        print('Дискрипторы не найдены в датасете')
        processed_descriptors = {}
        paths_df = pd.DataFrame()
        pca_results = {}
else:
    print('Дискрипторы ещё в памяти')

# Загрузка результатов кластеризации (если нужны)
if 'cluster_labels' not in dir():
    if os.path.exists(f'{data_path}/03/clustering_results.pickle'):
        with open(f'{data_path}/03/clustering_results.pickle', 'rb') as f:
            clustering_results = pickle.load(f)
        
        with open(f'{data_path}/03/cluster_labels.pickle', 'rb') as f:
            cluster_labels = pickle.load(f)
        
        print('Результаты кластеризации загружены из датасета')
    else:
        print('Результаты кластеризации не найдены в датасете')
        clustering_results = {}
        cluster_labels = {} 

print("\n" + "="*60)
print("\nЗАГРУЗКА ОКОНЧЕНА")
print("\n" + "="*60)

Дискрипторы ещё в памяти


ЗАГРУЗКА ОКОНЧЕНА



In [10]:
print("\n" + "="*60)
print("ПРОВЕРКА РАЗНЫХ КОЛИЧЕСТВ КЛАСТЕРОВ ДЛЯ EfficientNet")
print("="*60)

X_ef = processed_descriptors['EfficientNet']

# Проверим разные количества кластеров для K-Means
cluster_counts = [10, 15, 20, 25, 30, 40, 50]

print("\nСравнение метрик для разных k (EfficientNet):")
print("-" * 70)
print(f"{'k':<5} {'Calinski-Harabasz':<20} {'Davies-Bouldin':<20}")
print("-" * 70)

for k in cluster_counts:
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=10000, random_state=42, n_init=3)
    labels = kmeans.fit_predict(X_ef)
    
    ch_score = calinski_harabasz_score(X_ef, labels)
    db_score = davies_bouldin_score(X_ef, labels)
    
    print(f"{k:<5} {ch_score:<20.2f} {db_score:<20.2f}")

print("\n" + "="*60)


ПРОВЕРКА РАЗНЫХ КОЛИЧЕСТВ КЛАСТЕРОВ ДЛЯ EfficientNet

Сравнение метрик для разных k (EfficientNet):
----------------------------------------------------------------------
k     Calinski-Harabasz    Davies-Bouldin      
----------------------------------------------------------------------
10    5475.92              6.60                
15    3787.71              6.16                
20    2741.69              4.69                
25    2411.83              6.29                
30    2036.76              5.69                
40    1610.70              5.79                
50    1329.91              5.89                

